# Competencia: ¿qué le pidieron al asistente?

**Sesión 5 · IA Intermedio · EstudIA 2026**

Cada texto es algo que una persona le dijo a un asistente de voz, como “pon una alarma a las seis” o “qué tiempo
hace en Madrid”. Tu red tiene que decir la **intención**: qué quería la persona, entre **60 intenciones** (poner
una alarma, subir el volumen, pedir un domicilio, preguntar el clima…).

**La métrica: exactitud.** Esta vez sí usamos la exactitud de siempre. ¿Por qué no la balanceada? Porque ninguna
intención domina: la más frecuente es apenas el 7 % de los textos, así que responder siempre lo mismo da 7 %, y para
ganar hay que acertar en muchas intenciones a la vez.

**La línea que hay que alcanzar: una bolsa de palabras.** En la sección 1.2 la entrenas en segundos. ¿Tu red
recurrente la alcanza? ¿La supera?

**Reglas**

1. Tu modelo es una **red recurrente** (GRU o LSTM) en PyTorch. Puedes usar los vectores de palabras de fastText
   que trae el notebook. Nada de transformers ni modelos de lenguaje preentrenados: llegan en la sesión 6.
2. Tienes 13.547 textos con su intención y 2.974 sin intención. Separa tu propia validación: no tienes las
   respuestas de prueba.
3. Envía tus predicciones todas las veces que quieras (una vez cada 10 segundos). Gana la mayor **exactitud**.
4. La tabla en vivo usa **la mitad** de los textos de prueba. Al final, el profesor revela el resultado con **la
   otra mitad**: ese decide la competencia.

**Cómo empezar.** *Archivo › Guardar una copia en Drive*. No necesitas GPU: con CPU funciona. Ejecuta todo el
notebook (*Entorno de ejecución › Ejecutar todas*). Cuando te pida la clave, cópiala de la página de la competencia
en la plataforma y pégala. Así haces tu primer envío. Después, mejora tu red.

In [ ]:
# Preparación: librerías y datos. Ejecútala una vez.
import os, re, time, urllib.request
from collections import Counter
ARCHIVOS = os.environ.get("ESTUDIA_ARCHIVOS", "https://storage.googleapis.com/estud-ia-course-estudia-lab/s05")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

DISPOSITIVO = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(max(1, os.cpu_count() or 1))

def descargar(nombre, url=None):
    """Baja un archivo (de la carpeta pública del curso, o de `url`) solo la primera vez y devuelve su ruta."""
    ruta = os.path.join("archivos_curso", nombre)
    if not os.path.exists(ruta):
        os.makedirs(os.path.dirname(ruta), exist_ok=True)
        urllib.request.urlretrieve(url or f"{ARCHIVOS}/{nombre}", ruta)
    return ruta

def miles(n):
    """12345 -> "12.345", como escribimos los números en español."""
    return f"{n:,}".replace(",", ".")

def pct(x, decimales=1):
    """0.958 -> "95,8 %"."""
    return f"{100 * x:.{decimales}f} %".replace(".", ",")

def tokens(texto):
    """Parte un texto en palabras y signos, en minúsculas: "No me gustó." -> ["no", "me", "gustó", "."]."""
    return re.findall(r"\w+|[^\w\s]", str(texto).lower())

datos = pd.read_csv(descargar("asistente_entrenamiento.csv"))
prueba = pd.read_csv(descargar("asistente_prueba.csv"))
CLASES = sorted(datos.intencion.unique())
print(f"Listo: {miles(len(datos))} textos con intención, {miles(len(prueba))} de prueba, {len(CLASES)} intenciones.",
      "Con GPU." if DISPOSITIVO.type == "cuda" else "Sin GPU: funciona igual.")

In [ ]:
# @title Función para enviar tus predicciones (no hace falta cambiarla)
import getpass, json, urllib.error

PLATAFORMA = os.environ.get("ESTUDIA_PLATAFORMA", "https://estud-ia.eslopezag.com")
_CLAVE = os.environ.get("ESTUDIA_CLAVE", "")

def enviar(ids, predicciones, nota=""):
    """Envía tus predicciones a la plataforma del curso y muestra tu exactitud en la tabla.

    ids: los ids de las solicitudes de prueba. predicciones: una intención por solicitud, como "alarma_poner".
    nota: un texto corto para recordar qué probaste (opcional)."""
    global _CLAVE
    if not _CLAVE:
        _CLAVE = getpass.getpass("Pega tu clave (plataforma › Competencia › Copiar) y oprime Enter: ").strip()
    cuerpo = json.dumps({"ids": [int(i) for i in ids], "predicciones": [str(p) for p in predicciones],
                         "nota": str(nota)[:80]}).encode()
    req = urllib.request.Request(f"{PLATAFORMA}/api/competencia/s05/envios", data=cuerpo, method="POST",
                                 headers={"Content-Type": "application/json", "Authorization": f"Bearer {_CLAVE}",
                                          "User-Agent": "EstudIA-notebook/1.0"})
    try:
        with urllib.request.urlopen(req, timeout=60) as r:
            res = json.loads(r.read())
    except urllib.error.HTTPError as e:
        res = json.loads(e.read() or b"{}")
        if e.code == 401:
            _CLAVE = ""  # la próxima vez la pide de nuevo
        print("No se pudo enviar:", res.get("error", f"error {e.code}"))
        return
    except urllib.error.URLError:
        print("No hay conexión con la plataforma. Revisa tu internet o pregúntale al profesor si está encendida.")
        return
    metrica = res.get("metrica", "Exactitud")
    print(f"Envío {res['envio']} recibido. {metrica} en la tabla: {pct(res.get('puntaje', res['exactitud']))}.")
    print(f"Tu mejor resultado: {pct(res['mejor'])} · puesto {res['puesto']} de {res['participantes']} ({res['alias']}).")

In [ ]:
# @title Funciones para los vectores de palabras de fastText
_VECTORES = None

def vectores():
    """Las 100.000 palabras más frecuentes del español con su vector de 300 números (fastText). Se baja una vez."""
    global _VECTORES
    if _VECTORES is None:
        datos = np.load(descargar("vectores_es.npz"))
        matriz = datos["vectores"].astype(np.float32)
        matriz /= np.linalg.norm(matriz, axis=1, keepdims=True)
        palabras = datos["palabras"].tolist()
        _VECTORES = (palabras, {p: i for i, p in enumerate(palabras)}, matriz)
    return _VECTORES

def vector_de(palabra):
    """El vector de una palabra, o None si no está entre las 100.000. Prueba también con mayúscula inicial."""
    palabras, indice, matriz = vectores()
    for forma in (palabra, palabra.lower(), palabra.capitalize()):
        if forma in indice:
            return matriz[indice[forma]]
    return None

def vecinos(palabra, cuantos=6):
    """Las palabras con el vector más parecido (similitud del coseno)."""
    palabras, indice, matriz = vectores()
    v = vector_de(palabra)
    if v is None:
        return f"“{palabra}” no está entre las 100.000 palabras."
    orden = np.argsort(-(matriz @ v))
    return [palabras[i] for i in orden[:cuantos + 3] if palabras[i].lower() != palabra.lower()][:cuantos]

def analogia(a, b, c, cuantas=4):
    """b - a + c: “París es a Francia lo que ___ es a Colombia” es analogia("Francia", "París", "Colombia")."""
    palabras, indice, matriz = vectores()
    va, vb, vc = vector_de(a), vector_de(b), vector_de(c)
    q = vb - va + vc
    orden = np.argsort(-(matriz @ (q / np.linalg.norm(q))))
    fuera = {a.lower(), b.lower(), c.lower()}
    return [palabras[i] for i in orden[:20] if palabras[i].lower() not in fuera][:cuantas]

---
## 1. Los datos

In [ ]:
print(datos.sample(8, random_state=3).to_string(index=False))
cuantos = datos.intencion.value_counts()
print(f"\nTextos por intención: entre {cuantos.min()} ({cuantos.idxmin()}) y {cuantos.max()} ({cuantos.idxmax()}).")
print("Palabras por texto (mediana):", int(datos.texto.str.split().str.len().median()))

### 1.1 Entrenamiento y validación

Separamos el 10 % como validación, con las mismas proporciones de cada intención.

In [ ]:
ent, val = train_test_split(datos, test_size=0.1, stratify=datos.intencion, random_state=42)
print(f"Entrenamiento: {miles(len(ent))} · Validación: {miles(len(val))} · Prueba: {miles(len(prueba))}")

### 1.2 La línea que hay que alcanzar: una bolsa de palabras

Contamos palabras y pares de palabras (con pesos TF-IDF, que bajan el peso de las palabras que aparecen en todos
los textos) y entrenamos una regresión logística.

In [ ]:
bolsa = TfidfVectorizer(tokenizer=tokens, lowercase=False, token_pattern=None, ngram_range=(1, 2), sublinear_tf=True)
linea_base = LogisticRegression(max_iter=3000, C=10).fit(bolsa.fit_transform(ent.texto), ent.intencion)
LINEA_BASE = (linea_base.predict(bolsa.transform(val.texto)) == val.intencion).mean()
print("Bolsa de palabras · exactitud de validación:", pct(LINEA_BASE))

---
## 2. Tu red

Las piezas que puedes cambiar:

- `TIPO`: `"GRU"` o `"LSTM"`. `BIDIRECCIONAL`: si es `True`, otra red lee de derecha a izquierda.
- `VECTORES`: si es `True`, la red empieza con los vectores de fastText (300 números por palabra). `CONGELAR`:
  si es `True`, esos vectores no cambian durante el entrenamiento.
- `DIM` (números por palabra si no usas fastText), `OCULTA` (tamaño del estado oculto) y `DROPOUT`.
- `EPOCAS`, `TASA` y `LOTE`.

Así como vienen funcionan, pero dejan mucho por ganar. Ejecútalas tal cual para tu primer envío y después mejóralas.

In [ ]:
TIPO = "GRU"
BIDIRECCIONAL = False
VECTORES = False
CONGELAR = True
DIM = 64
OCULTA = 64
DROPOUT = 0.0
EPOCAS = 5
TASA = 2e-3
LOTE = 64
LARGO = 20      # palabras por texto; casi todos tienen menos

In [ ]:
# @title La red recurrente (la misma de la demostración)
class RedRecurrente(nn.Module):
    """Embedding -> GRU (o LSTM) -> una capa con una salida por clase."""
    def __init__(self, n_palabras, n_clases, dim=128, oculta=128, bidireccional=True, tipo="GRU",
                 pesos=None, congelar=False, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(n_palabras, dim, padding_idx=0)
        if pesos is not None:                      # vectores preentrenados
            self.embedding.weight.data.copy_(torch.as_tensor(pesos))
            self.embedding.weight.requires_grad = not congelar
        capa = nn.GRU if tipo == "GRU" else nn.LSTM
        self.rnn = capa(dim, oculta, batch_first=True, bidirectional=bidireccional)
        self.dropout = nn.Dropout(dropout)
        self.salida = nn.Linear(oculta * (2 if bidireccional else 1), n_clases)

    def forward(self, x):
        largos = (x != 0).sum(1).clamp(min=1).cpu()
        e = self.dropout(self.embedding(x))
        paquete = nn.utils.rnn.pack_padded_sequence(e, largos, batch_first=True, enforce_sorted=False)
        _, h = self.rnn(paquete)
        if isinstance(h, tuple):                   # la LSTM devuelve (h, c)
            h = h[0]
        h = torch.cat(list(h), dim=1)              # el último estado de cada dirección
        return self.salida(self.dropout(h))

In [ ]:
conteo = Counter(w for t in ent.texto for w in tokens(t))
VOCAB = {"<relleno>": 0, "<desconocida>": 1}
for palabra, veces in conteo.most_common():
    if veces >= 2:
        VOCAB[palabra] = len(VOCAB)

def a_matriz(textos):
    m = np.zeros((len(textos), LARGO), dtype=np.int64)
    for i, t in enumerate(textos):
        ids = [VOCAB.get(w, 1) for w in tokens(t)][:LARGO]
        m[i, :len(ids)] = ids
    return torch.from_numpy(m)

def pesos_fasttext():
    """Una fila por palabra del vocabulario: su vector de fastText, o números al azar si no lo tiene."""
    pesos = np.random.default_rng(0).normal(0, 0.1, (len(VOCAB), 300)).astype(np.float32)
    pesos[0] = 0
    for palabra, i in VOCAB.items():
        v = vector_de(palabra) if i > 1 else None
        if v is not None:
            pesos[i] = v
    return pesos

def crear_modelo():
    if VECTORES:
        return RedRecurrente(len(VOCAB), len(CLASES), dim=300, oculta=OCULTA, bidireccional=BIDIRECCIONAL, tipo=TIPO,
                             pesos=pesos_fasttext(), congelar=CONGELAR, dropout=DROPOUT)
    return RedRecurrente(len(VOCAB), len(CLASES), dim=DIM, oculta=OCULTA, bidireccional=BIDIRECCIONAL, tipo=TIPO,
                         dropout=DROPOUT)

def entrenar(textos, intenciones, textos_val=None, intenciones_val=None):
    """Entrena una red nueva. Si le das validación, la mide al final de cada época."""
    torch.manual_seed(0)
    X, y = a_matriz(list(textos)), torch.tensor([CLASES.index(c) for c in intenciones])
    modelo = crear_modelo().to(DISPOSITIVO)
    optimizador = torch.optim.Adam([p for p in modelo.parameters() if p.requires_grad], lr=TASA)
    historia, inicio = [], time.time()
    for epoca in range(EPOCAS):
        modelo.train()
        orden = torch.randperm(len(X))
        for i in range(0, len(X), LOTE):
            idx = orden[i:i + LOTE]
            perdida = F.cross_entropy(modelo(X[idx].to(DISPOSITIVO)), y[idx].to(DISPOSITIVO))
            optimizador.zero_grad(); perdida.backward(); optimizador.step()
        if textos_val is not None:
            historia.append((predecir(modelo, textos_val) == np.array(list(intenciones_val))).mean())
    print(f"{EPOCAS} épocas en {time.time() - inicio:.0f} s.")
    return modelo, historia

@torch.no_grad()
def predecir(modelo, textos):
    modelo.eval()
    X = a_matriz(list(textos))
    salida = torch.cat([modelo(X[i:i + 1000].to(DISPOSITIVO)).argmax(1).cpu() for i in range(0, len(X), 1000)])
    return np.array([CLASES[i] for i in salida])

### 2.1 Entrenar y validar

In [ ]:
modelo, historia = entrenar(ent.texto, ent.intencion, val.texto, val.intencion)
plt.figure(figsize=(7, 3))
plt.plot(range(1, len(historia) + 1), historia, color="#CB318E", lw=2, label="Tu red")
plt.axhline(LINEA_BASE, color="#7FC200", ls="--", lw=1.5, label=f"Bolsa de palabras ({pct(LINEA_BASE)})")
plt.xlabel("Época"); plt.ylabel("Exactitud en validación"); plt.ylim(0, 1); plt.legend(frameon=False); plt.grid(alpha=0.3)
plt.show()
print(f"Tu red · exactitud de validación: {pct(historia[-1])} · bolsa de palabras: {pct(LINEA_BASE)}")

### 2.2 Predecir la prueba y enviar

Cambia la nota para recordar qué probaste en cada envío. La primera vez te pide la clave.

In [ ]:
predicciones = predecir(modelo, prueba.texto)
enviar(prueba.id, predicciones, nota="primer envío")

---
## 3. Pistas para subir en la tabla

Prueba una cosa a la vez, mira la exactitud de **validación** y envía cuando mejore.

<details><summary>Pista 1</summary>

Mira la curva: ¿la exactitud de validación seguía subiendo en la última época? Entonces la red necesita más
épocas.
</details>

<details><summary>Pista 2</summary>

`BIDIRECCIONAL = True`: una segunda red lee el texto de derecha a izquierda. En “apaga la luz de la sala”, la palabra
que decide puede estar al principio o al final.
</details>

<details><summary>Pista 3</summary>

`VECTORES = True`: la red empieza con los vectores de fastText, que ya saben que “apaga” y “apagar” se parecen, o
que “taxi” y “uber” también. La primera vez se bajan (unos 55 MB). Prueba con `CONGELAR = True` y con `False`.
</details>

<details><summary>Pista 4</summary>

Más capacidad y más freno: sube `OCULTA` (por ejemplo, a 128 o 256) y pon algo de `DROPOUT` (0,3) para que no
memorice. Prueba también `TIPO = "LSTM"`.
</details>

<details><summary>Pista 5</summary>

Cuando ya elegiste todo, entrena el modelo final con todos los textos (entrenamiento más validación) antes de
predecir la prueba: `modelo, _ = entrenar(datos.texto, datos.intencion)`.
</details>

**Una advertencia.** Si eliges tus cambios mirando solo la tabla en vivo, terminas ajustando el modelo a esa mitad
de la prueba. El resultado final se calcula con la otra mitad.

**Fuente de los datos:** FitzGerald, J. y otros (2023). MASSIVE: A 1M-Example Multilingual Natural Language
Understanding Dataset with 51 Typologically-Diverse Languages. *ACL*. Licencia CC BY 4.0. Usamos los textos en
español (es-ES) y tradujimos los nombres de las intenciones. Vectores de palabras: fastText (Grave y otros, 2018),
CC BY-SA 3.0.